
# 04 — Robustez, sensibilidad y confiabilidad del modelo RADAR Cibest | Deep Dive Ejecutivo

**Fase ASUM-DM:** 5 — Evaluación  
**Proyecto:** RADAR Cibest — Ranking de Atractivo y Diagnóstico Analítico Regional  
**Propósito:** evaluar si las prioridades país/línea son suficientemente robustas para soportar decisiones ejecutivas bajo incertidumbre de pesos, método, mezcla RADAR y variabilidad del ranking.

---

## Tesis central

El Notebook 04 no debe responder únicamente si el ranking cambia cuando se modifican pesos. Debe responder una pregunta más exigente:

> **¿Qué tan confiable es la recomendación del modelo RADAR para priorizar países y líneas de negocio ante incertidumbre razonable en los supuestos?**

La respuesta se construye en tres niveles:

1. **Robustez estructural TOPSIS:** estabilidad del atractivo país/línea cuando se perturban pesos de dimensiones y variables.
2. **Robustez decisional RADAR:** estabilidad del score final usado para decisión cuando se perturban TOPSIS y la mezcla `alpha`, `beta`, `gamma`.
3. **Confiabilidad ejecutiva:** probabilidad de mantenerse en Top-N, probabilidad de permanecer en banda alta, volatilidad de ranking y consistencia entre líneas.

La decisión final debe privilegiar **Monte Carlo RADAR**, porque RADAR es la métrica compuesta usada para priorización:

```text
RADAR = alpha * TOPSIS + beta * IPC + gamma * Trend
```

---

## Cómo debe leerse el notebook ante Comité

- Un país con buen ranking base pero alta volatilidad no es una prioridad robusta; es una prioridad condicional.
- Un país que no siempre es Top 5 pero permanece en banda alta/media-alta puede ser una oportunidad exploratoria defendible.
- Una correlación alta entre líneas no implica error si los pesos son distintos; puede indicar un factor país común.
- La robustez no prueba éxito comercial; prueba estabilidad interna de la recomendación ante incertidumbre razonable del modelo.



## 1. Configuración reproducible y recarga profunda de módulos

Esta celda evita que el notebook use versiones antiguas de `src/` después de cambios en `cleaning.py`, `hybrid_scorer.py`, `monte_carlo.py` o configuración de pesos. La recarga debe ejecutarse antes de importar funciones sueltas.


In [ ]:

from __future__ import annotations

import sys
import re
import importlib
import inspect
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import src.utils as utils
import src.data_preparation.cleaning as cleaning
import src.data_preparation.feature_engineering as feature_engineering
import src.scoring.hybrid_scorer as hybrid_scorer
import src.scoring.sensitivity as sensitivity
import src.scoring.monte_carlo as monte_carlo
import src.scoring.weighting as weighting

importlib.invalidate_caches()
importlib.reload(utils)
importlib.reload(cleaning)
importlib.reload(feature_engineering)
importlib.reload(weighting)
importlib.reload(sensitivity)
importlib.reload(monte_carlo)
importlib.reload(hybrid_scorer)

from src.utils import load_all_configs, setup_logger, resolve_data_path, get_variable_catalog
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring
from src.scoring.weighting import compute_hierarchical_weights
from src.scoring.sensitivity import run_sensitivity_analysis, compare_topsis_vikor
from src.scoring.monte_carlo import (
    coerce_component_series,
    run_monte_carlo_topsis_robustness,
    run_monte_carlo_radar_robustness,
)

configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))
variable_catalog = get_variable_catalog(configs["variables"])

print("cleaning file:", cleaning.__file__)
print("hybrid_scorer file:", hybrid_scorer.__file__)
print("monte_carlo file:", monte_carlo.__file__)
print("run_cleaning line:", inspect.getsourcelines(cleaning.run_cleaning)[1])
print("run_monte_carlo_radar_robustness line:", inspect.getsourcelines(monte_carlo.run_monte_carlo_radar_robustness)[1])


2026-06-26 01:19:06 | INFO     | src.utils:setup_logger:174 | File logging desactivado. Solo consola activa.


cleaning file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\data_preparation\cleaning.py
hybrid_scorer file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\scoring\hybrid_scorer.py
monte_carlo file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\scoring\monte_carlo.py
run_cleaning line: 355
run_monte_carlo_radar_robustness line: 679


2026-06-26 01:19:06 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 46 variables (estrategia=latest_available)
2026-06-26 01:19:06 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=52
2026-06-26 01:19:06 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'domestic_credit_private_gdp': 2, 'personal_remittances_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'trade_openness': 2, 'current_account_gdp': 2, 'digital_payment_adults_pct': 2, 'return_on_equity': 1, 'gdp_per_capita_ppp': 1}
2026-06-26 01:19:06 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:230 | Países con más 


## 2. Estilo Cibest y utilidades de interpretación

La lectura de robustez debe ser visualmente sobria y orientada a decisión. Las tablas se enfocan en probabilidades, volatilidad, bandas y clasificación ejecutiva.


In [2]:

CIBEST = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
}

TIER_COLORS = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}


def style_table(
    df: pd.DataFrame,
    gradient_cols: Optional[List[str]] = None,
    gradient_cmap: str = "YlGnBu",
    format_dict: Optional[Dict[str, str]] = None,
):
    """Aplica estilo Cibest a tablas pandas."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        cols = [c for c in gradient_cols if c in df.columns]
        if cols:
            styler = styler.background_gradient(subset=cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str) -> None:
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str) -> None:
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str) -> None:
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))

success_box("Entorno listo", "Notebook de robustez preparado para evaluar confiabilidad ejecutiva del ranking RADAR.")



## 3. Carga del master vigente y construcción de matriz de decisión

La robustez solo tiene sentido si se evalúa sobre el mismo universo que alimenta el scoring oficial. Esta sección valida:

- master raw vigente;
- exclusiones por calidad;
- ausencia de `gdp_growth` en TOPSIS;
- consistencia de `decision_matrix`.


In [3]:

raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

master_files = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not master_files:
    raise FileNotFoundError("No se encontró master_raw_YYYYMMDD.parquet. Ejecute primero el notebook 01.")

master_path = master_files[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

wide_raw, decision_matrix, excluded_countries = prepare_decision_matrix(master, configs)

master_summary = pd.DataFrame({
    "métrica": ["archivo", "filas", "países_master", "variables_master", "wide_raw", "decision_matrix", "países_excluidos", "gdp_growth_en_decision_matrix"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        str(wide_raw.shape),
        str(decision_matrix.shape),
        ", ".join(sorted(excluded_countries)) if excluded_countries else "Ninguno",
        "Sí" if "gdp_growth" in decision_matrix.columns else "No",
    ],
})

display(style_table(master_summary))

if "gdp_growth" in decision_matrix.columns:
    raise ValueError("gdp_growth no debe estar en decision_matrix. Debe alimentar solo Trend.")

if excluded_countries:
    risk_box(
        "Exclusiones por calidad aplicadas",
        "Estos países no entran al universo de simulación. Esto reduce sesgo por imputación excesiva y evita que TOPSIS construya ideales con alternativas de baja cobertura: "
        f"{', '.join(sorted(excluded_countries))}."
    )


,métrica,valor
0,archivo,master_raw_20260626.parquet
1,filas,1318
2,países_master,30
3,variables_master,46
4,wide_raw,"(25, 47)"
5,decision_matrix,"(25, 36)"
6,países_excluidos,"BRB, CUB, GUY, HTI, VEN"
7,gdp_growth_en_decision_matrix,No



## 4. Sensibilidad determinística: prueba rápida de fragilidad estructural

La sensibilidad determinística evalúa perturbaciones puntuales de pesos. Sirve como diagnóstico inicial, pero no reemplaza Monte Carlo porque no explora una distribución completa de escenarios.

**Lectura ejecutiva:** si la correlación de scores/ranking se mantiene alta y el Top 10 cambia poco, el ranking estructural no depende excesivamente de cambios marginales de pesos.


In [4]:

sens = run_sensitivity_analysis(
    decision_matrix=decision_matrix,
    dimension_weights=configs["weights"]["dimension_weights"],
    variable_weights_by_dim=configs["weights"]["variable_weights"],
    variable_catalog=variable_catalog,
)

sens_df = sens if isinstance(sens, pd.DataFrame) else pd.DataFrame(sens)
display(style_table(sens_df, gradient_cols=[c for c in sens_df.columns if "corr" in c.lower() or "overlap" in c.lower()], gradient_cmap="RdYlGn"))

# Interpretación automática si existen columnas estándar.
possible_corr_cols = [c for c in sens_df.columns if "corr" in c.lower() or "spearman" in c.lower()]
possible_overlap_cols = [c for c in sens_df.columns if "overlap" in c.lower()]

if possible_corr_cols:
    corr_col = possible_corr_cols[0]
    mean_corr = pd.to_numeric(sens_df[corr_col], errors="coerce").mean()
else:
    mean_corr = np.nan

if possible_overlap_cols:
    overlap_col = possible_overlap_cols[0]
    mean_overlap = pd.to_numeric(sens_df[overlap_col], errors="coerce").mean()
else:
    mean_overlap = np.nan

if pd.notna(mean_corr) and mean_corr >= 0.90:
    success_box(
        "Sensibilidad determinística favorable",
        f"La correlación promedio ante perturbaciones puntuales es alta ({mean_corr:.3f}). Esto sugiere que el ranking estructural no depende críticamente de un único vector de pesos."
    )
elif pd.notna(mean_corr):
    risk_box(
        "Sensibilidad determinística relevante",
        f"La correlación promedio es {mean_corr:.3f}. Conviene revisar qué dimensiones o variables generan mayor inestabilidad antes de presentar conclusiones cerradas."
    )


,dimension,perturbation,score_corr,topN_overlap,topN_size,countries_in,countries_out
0,macro,0.800000,0.994600,10,10,,
1,macro,0.900000,0.995400,10,10,,
2,macro,1.100000,0.993800,10,10,,
3,macro,1.200000,0.988500,9,10,MEX,PAN
4,financial,0.800000,0.990800,10,10,,
5,financial,0.900000,0.995400,10,10,,
6,financial,1.100000,0.996200,10,10,,
7,financial,1.200000,0.991500,10,10,,
8,institutional,0.800000,0.980000,9,10,MEX,BHS
9,institutional,0.900000,0.993800,10,10,,



## 5. TOPSIS vs VIKOR: robustez metodológica del enfoque multicriterio

TOPSIS y VIKOR no resuelven el problema de la misma forma. TOPSIS prioriza cercanía relativa al ideal positivo y distancia al ideal negativo; VIKOR introduce una lógica de compromiso.

**Pregunta para Comité:** si otro método multicriterio razonable produce resultados parecidos, la recomendación gana credibilidad metodológica.


In [5]:

var_weights = compute_hierarchical_weights(
    configs["weights"]["dimension_weights"],
    configs["weights"]["variable_weights"],
)
var_weights = {variable: weight for variable, weight in var_weights.items() if variable in decision_matrix.columns}

comparison = compare_topsis_vikor(decision_matrix, var_weights, variable_catalog)

display(style_table(
    comparison.head(20),
    gradient_cols=[col for col in comparison.columns if "rank" in col.lower() or "score" in col.lower()],
    gradient_cmap="RdYlGn",
).set_caption("Comparación TOPSIS vs VIKOR"))

rank_cols = [c for c in comparison.columns if "rank" in c.lower()]
if len(rank_cols) >= 2:
    method_corr = comparison[rank_cols[0]].corr(comparison[rank_cols[1]], method="spearman")
    if method_corr >= 0.85:
        success_box(
            "Robustez metodológica alta",
            f"La correlación Spearman entre métodos es {method_corr:.3f}. Esto indica que el ranking no depende fuertemente de una única técnica multicriterio."
        )
    elif method_corr >= 0.70:
        insight_box(
            "Robustez metodológica moderada",
            f"La correlación Spearman entre métodos es {method_corr:.3f}. Las prioridades principales pueden ser defendibles, pero conviene revisar países con diferencias de ranking amplias."
        )
    else:
        risk_box(
            "Sensibilidad metodológica alta",
            f"La correlación Spearman entre métodos es {method_corr:.3f}. El Comité debe conocer que la recomendación depende de la elección metodológica."
        )


,score_topsis,rank_topsis,score_vikor,rank_vikor,rank_diff
country_iso3,,,,,
CAN,0.708873,1,0.981949,2,1
USA,0.707668,2,1.000000,1,1
CHL,0.620973,3,0.724375,3,0
ESP,0.618113,4,0.697068,4,0
URY,0.587365,5,0.664998,5,0
CRI,0.527153,6,0.546331,7,1
BRA,0.525710,7,0.435176,11,4
BHS,0.518497,8,0.401766,12,4
DOM,0.514112,9,0.469695,8,1



## 6. Escenario base RADAR: punto central de comparación

Monte Carlo no reemplaza el ranking base. Lo convierte en una distribución probabilística. El ranking base es el escenario central; Monte Carlo responde qué tanto debemos confiar en él.


In [6]:

results = run_full_scoring(master, configs, persist=True)

ipc_scores = coerce_component_series(results["ipc"], value_col=None, component_name="ipc")
trend_scores = coerce_component_series(results["trend"], value_col="trend", component_name="trend")

base_summary = pd.DataFrame({
    "métrica": ["países_RADAR", "IPC_disponible", "Trend_disponible", "alpha", "beta", "gamma", "excluidos"],
    "valor": [
        len(results["radar_global"]),
        ipc_scores.notna().sum(),
        trend_scores.notna().sum(),
        results["composite_weights"]["alpha"],
        results["composite_weights"]["beta"],
        results["composite_weights"]["gamma"],
        ", ".join(sorted(results.get("excluded_countries", []))) if results.get("excluded_countries") else "Ninguno",
    ],
})

display(style_table(base_summary))

insight_box(
    "Interpretación del escenario base",
    "El ranking base es la recomendación determinística. La robustez se evalúa observando qué países preservan su posición, banda o pertenencia al Top-N cuando perturbamos pesos estructurales y pesos compuestos."
)


,métrica,valor
0,países_RADAR,24
1,IPC_disponible,24
2,Trend_disponible,24
3,alpha,0.600000
4,beta,0.300000
5,gamma,0.100000
6,excluidos,"BRB, CUB, GUY, HTI, VEN"



## 7. Monte Carlo TOPSIS: robustez estructural por línea

Monte Carlo TOPSIS perturba pesos de dimensiones y variables. Responde:

> **¿El atractivo estructural de cada país/línea se mantiene cuando cambia razonablemente la importancia relativa de los criterios?**

No incluye IPC ni Trend. Por eso no debe usarse como prueba final de decisión, sino como prueba de estabilidad del componente estructural.


In [7]:

mc_cfg = configs["settings"].get("monte_carlo", {})

mc_topsis = run_monte_carlo_topsis_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=variable_catalog,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(mc_cfg.get("topsis", {}).get("dimension_concentration", 150)),
    variable_concentration=float(mc_cfg.get("topsis", {}).get("variable_concentration", 100)),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

mc_topsis_summary = pd.DataFrame({
    "métrica": ["filas_simuladas", "líneas", "países", "simulaciones"],
    "valor": [
        mc_topsis["simulation_long"].shape[0],
        mc_topsis["simulation_long"]["business_line"].nunique(),
        mc_topsis["simulation_long"]["country_iso3"].nunique(),
        mc_topsis["simulation_long"]["simulation_id"].nunique(),
    ],
})

display(style_table(mc_topsis_summary))


,métrica,valor
0,filas_simuladas,120000
1,líneas,5
2,países,24
3,simulaciones,1000



## 8. Monte Carlo RADAR: prueba principal de confiabilidad decisional

Monte Carlo RADAR es la prueba más relevante porque evalúa la métrica final usada para decidir. Perturba:

- pesos TOPSIS de dimensiones;
- pesos TOPSIS de variables;
- mezcla compuesta `alpha`, `beta`, `gamma`, si está configurado.

**Pregunta ejecutiva:** ¿los países recomendados siguen siendo atractivos cuando movemos supuestos razonablemente?


In [8]:

mc_radar = run_monte_carlo_radar_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=variable_catalog,
    ipc_scores=ipc_scores,
    trend_scores=trend_scores,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(mc_cfg.get("topsis", {}).get("dimension_concentration", 150)),
    variable_concentration=float(mc_cfg.get("topsis", {}).get("variable_concentration", 100)),
    composite_concentration=float(mc_cfg.get("radar", {}).get("composite_concentration", 150)),
    perturb_composite_weights=bool(mc_cfg.get("radar", {}).get("perturb_composite_weights", True)),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

mc_radar_summary = pd.DataFrame({
    "métrica": ["filas_simuladas", "líneas", "países", "simulaciones"],
    "valor": [
        mc_radar["simulation_long"].shape[0],
        mc_radar["simulation_long"]["business_line"].nunique(),
        mc_radar["simulation_long"]["country_iso3"].nunique(),
        mc_radar["simulation_long"]["simulation_id"].nunique(),
    ],
})

display(style_table(mc_radar_summary))

expected_rows = (
    int(mc_cfg.get("n_simulations", 1000))
    * mc_radar["simulation_long"]["business_line"].nunique()
    * mc_radar["simulation_long"]["country_iso3"].nunique()
)

if mc_radar["simulation_long"].shape[0] == expected_rows:
    success_box("Monte Carlo RADAR completo", f"Se generaron {expected_rows:,} observaciones país-línea-escenario.")
else:
    risk_box("Revisar tamaño de simulación", f"Se esperaban {expected_rows:,} filas y se obtuvieron {mc_radar['simulation_long'].shape[0]:,}.")


,métrica,valor
0,filas_simuladas,120000
1,líneas,5
2,países,24
3,simulaciones,1000


In [9]:
sim_long = mc_radar["simulation_long"].copy()

# Agregación a nivel país-simulación
radar_global_sim = (
    sim_long
    .groupby(["simulation_id", "country_iso3"])
    .agg(
        radar_score_global_sim=("radar_score", "mean")  # o weighted mean si aplica
    )
    .reset_index()
)

radar_global_sim["rank_global_sim"] = (
    radar_global_sim
    .groupby("simulation_id")["radar_score_global_sim"]
    .rank(ascending=False, method="min")
)

global_robustness = (
    radar_global_sim
    .groupby("country_iso3")
    .agg(
        mean_rank=("rank_global_sim", "mean"),
        std_rank=("rank_global_sim", "std"),
        prob_top_3=("rank_global_sim", lambda x: (x <= 3).mean()),
        prob_top_5=("rank_global_sim", lambda x: (x <= 5).mean()),
        prob_top_10=("rank_global_sim", lambda x: (x <= 10).mean()),
    )
    .reset_index()
)

global_base = results["radar_global"].copy()

global_exec = (
    global_base
    .merge(global_robustness, on="country_iso3", how="left")
    .sort_values("radar_score", ascending=False)
)




## 9. Clasificación de robustez: de ranking puntual a decisión probabilística

La robustez se interpreta con cuatro variables:

1. **Ranking medio:** posición esperada bajo escenarios.
2. **Volatilidad del ranking:** sensibilidad ante pesos.
3. **Probabilidad Top-N:** probabilidad de seguir entre los países prioritarios.
4. **Probabilidad por banda:** estabilidad de atractividad, aunque la posición exacta cambie.

La clasificación ejecutiva recomendada es:

- **Prioridad robusta:** alta probabilidad de Top 5 y banda alta.
- **Prioridad condicional:** alta probabilidad de Top 10 y banda alta/media-alta.
- **Alta sensibilidad:** volatilidad elevada; requiere revisión de drivers.
- **No robusta:** buen ranking base insuficiente o baja estabilidad probabilística.


In [10]:

radar_rank_robustness = mc_radar["rank_robustness"]
radar_topn = mc_radar["topn_probabilities"]
radar_tiers = mc_radar["tier_probabilities"]

radar_robustness_exec = (
    radar_rank_robustness
    .merge(radar_topn, on=["business_line", "country_iso3"], how="left")
    .merge(radar_tiers, on=["business_line", "country_iso3"], how="left")
)

for col in ["Alta", "Media-alta", "Media", "Baja"]:
    if col not in radar_robustness_exec.columns:
        radar_robustness_exec[col] = 0.0


def classify_rank_robustness(row: pd.Series) -> str:
    """Clasifica robustez ejecutiva país-línea."""
    prob_top_5 = row.get("prob_top_5", 0.0)
    prob_top_10 = row.get("prob_top_10", 0.0)
    prob_high = row.get("Alta", 0.0)
    prob_high_mid = row.get("Alta", 0.0) + row.get("Media-alta", 0.0)
    std_rank = row.get("std_rank", 999.0)

    if prob_top_5 >= 0.80 and prob_high >= 0.70:
        return "Prioridad robusta"
    if prob_top_10 >= 0.70 and prob_high_mid >= 0.70:
        return "Prioridad condicional"
    if std_rank >= 4:
        return "Alta sensibilidad"
    return "Prioridad no robusta"


def confidence_label(row: pd.Series) -> str:
    """Etiqueta de confiabilidad para comunicación ejecutiva."""
    if row["robustness_class"] == "Prioridad robusta":
        return "Alta confiabilidad"
    if row["robustness_class"] == "Prioridad condicional":
        return "Confiabilidad media"
    if row["robustness_class"] == "Alta sensibilidad":
        return "Baja confiabilidad sin validación adicional"
    return "No prioritario / baja evidencia"

radar_robustness_exec["robustness_class"] = radar_robustness_exec.apply(classify_rank_robustness, axis=1)
radar_robustness_exec["confidence_label"] = radar_robustness_exec.apply(confidence_label, axis=1)

robustness_counts = (
    radar_robustness_exec
    .groupby(["business_line", "robustness_class"])
    .size()
    .rename("n_countries")
    .reset_index()
)

display(style_table(robustness_counts, gradient_cols=["n_countries"], format_dict={"n_countries": "{:.0f}"}))

fig = px.bar(
    robustness_counts,
    x="business_line",
    y="n_countries",
    color="robustness_class",
    title="Clasificación de países según robustez Monte Carlo RADAR",
    color_discrete_map={
        "Prioridad robusta": CIBEST["green"],
        "Prioridad condicional": CIBEST["gold"],
        "Alta sensibilidad": CIBEST["amber"],
        "Prioridad no robusta": CIBEST["red"],
    },
)
fig.update_layout(xaxis_title="Línea de negocio", yaxis_title="Número de países")
fig.show()


,business_line,robustness_class,n_countries
0,AD,Prioridad condicional,5
1,AD,Prioridad no robusta,15
2,AD,Prioridad robusta,4
3,BD,Prioridad condicional,3
4,BD,Prioridad no robusta,17
5,BD,Prioridad robusta,4
6,CIB,Prioridad condicional,5
7,CIB,Prioridad no robusta,16
8,CIB,Prioridad robusta,3
9,IB,Prioridad condicional,2



## 10. Top países robustos por línea: probabilidades, no solo posiciones

Esta tabla es la salida más importante para Comité. Permite responder:

- ¿Qué país tiene mayor probabilidad de permanecer en Top 5?
- ¿Qué país es Top 10 robusto aunque no siempre sea Top 5?
- ¿Qué países tienen ranking alto pero baja confiabilidad?


In [11]:
global_exec["robustness_class"] = global_exec.apply(classify_rank_robustness, axis=1)
global_exec["confidence_label"] = global_exec.apply(confidence_label, axis=1)

In [12]:
print(
    radar_global_sim.groupby("simulation_id")["radar_score_global_sim"].count().unique()
)

[24]


In [13]:
print(global_robustness.isna().sum())

country_iso3    0
mean_rank       0
std_rank        0
prob_top_3      0
prob_top_5      0
prob_top_10     0
dtype: int64


In [14]:

for business_line in sorted(radar_robustness_exec["business_line"].unique()):
    tmp = (
        radar_robustness_exec
        .query("business_line == @business_line")
        .sort_values(["prob_top_5", "prob_top_10", "Alta", "mean_rank"], ascending=[False, False, False, True])
        .head(15)
    )
    display(Markdown(f"### {business_line} — Top probabilístico RADAR"))
    display(style_table(
        tmp[[
            "country_iso3", "mean_rank", "median_rank", "std_rank", "p10_rank", "p90_rank",
            "prob_top_3", "prob_top_5", "prob_top_10", "Alta", "Media-alta",
            "robustness_class", "confidence_label",
        ]],
        gradient_cols=["mean_rank", "std_rank", "prob_top_5", "prob_top_10", "Alta"],
        gradient_cmap="RdYlGn",
        format_dict={
            "mean_rank": "{:.2f}",
            "median_rank": "{:.2f}",
            "std_rank": "{:.2f}",
            "p10_rank": "{:.1f}",
            "p90_rank": "{:.1f}",
            "prob_top_3": "{:.1%}",
            "prob_top_5": "{:.1%}",
            "prob_top_10": "{:.1%}",
            "Alta": "{:.1%}",
            "Media-alta": "{:.1%}",
        },
    ))


### AD — Top probabilístico RADAR

,country_iso3,mean_rank,median_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,CRI,1.67,2.00,0.64,1.0,2.0,98.2%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
2,ESP,2.88,3.00,0.52,2.0,3.0,93.6%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
1,PAN,1.73,1.00,1.00,1.0,3.0,92.4%,99.6%,100.0%,98.6%,1.4%,Prioridad robusta,Alta confiabilidad
3,USA,4.02,4.00,0.97,3.0,5.0,15.7%,93.4%,100.0%,80.3%,19.7%,Prioridad robusta,Alta confiabilidad
4,CHL,5.08,5.00,0.55,4.0,6.0,0.0%,82.4%,100.0%,10.4%,89.6%,Prioridad condicional,Confiabilidad media
5,DOM,6.36,6.00,1.27,4.0,8.0,0.1%,22.7%,100.0%,10.6%,89.4%,Prioridad condicional,Confiabilidad media
6,URY,6.58,7.00,0.52,6.0,7.0,0.0%,1.0%,100.0%,0.0%,100.0%,Prioridad condicional,Confiabilidad media
7,CAN,7.83,8.00,0.88,7.0,8.0,0.0%,0.8%,98.6%,0.1%,97.4%,Prioridad condicional,Confiabilidad media
8,BHS,9.70,9.00,1.49,9.0,12.0,0.0%,0.1%,80.8%,0.0%,70.2%,Prioridad condicional,Confiabilidad media
9,PER,10.17,10.00,0.92,9.0,11.0,0.0%,0.0%,69.6%,0.0%,22.7%,Prioridad no robusta,No prioritario / baja evidencia


### BD — Top probabilístico RADAR

,country_iso3,mean_rank,median_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
25,ESP,2.26,2.00,1.03,1.0,4.0,87.7%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
26,CRI,2.34,2.00,0.67,2.0,3.0,97.5%,100.0%,100.0%,99.6%,0.4%,Prioridad robusta,Alta confiabilidad
24,PAN,1.74,1.00,1.12,1.0,3.0,91.0%,99.1%,100.0%,96.6%,3.4%,Prioridad robusta,Alta confiabilidad
27,DOM,3.94,4.00,0.77,3.0,5.0,19.1%,96.2%,100.0%,86.0%,14.0%,Prioridad robusta,Alta confiabilidad
28,CHL,5.05,5.00,0.68,4.0,6.0,2.3%,82.2%,100.0%,12.2%,87.8%,Prioridad condicional,Confiabilidad media
29,USA,6.38,6.00,1.59,5.0,9.0,2.3%,19.5%,97.2%,5.5%,88.6%,Prioridad condicional,Confiabilidad media
30,MEX,7.50,7.00,1.15,6.0,9.0,0.1%,2.4%,98.3%,0.1%,93.4%,Prioridad condicional,Confiabilidad media
32,ECU,9.07,9.00,2.02,6.0,12.0,0.0%,0.6%,74.0%,0.0%,59.3%,Prioridad no robusta,No prioritario / baja evidencia
33,PER,9.27,9.00,1.12,8.0,11.0,0.0%,0.0%,87.3%,0.0%,60.3%,Prioridad no robusta,No prioritario / baja evidencia
31,URY,8.85,9.00,1.72,7.0,11.0,0.0%,0.0%,81.3%,0.0%,59.6%,Prioridad no robusta,No prioritario / baja evidencia


### CIB — Top probabilístico RADAR

,country_iso3,mean_rank,median_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
48,PAN,1.02,1.00,0.18,1.0,1.0,99.9%,100.0%,100.0%,99.9%,0.1%,Prioridad robusta,Alta confiabilidad
49,DOM,2.45,2.00,0.63,2.0,3.0,94.2%,100.0%,100.0%,99.5%,0.5%,Prioridad robusta,Alta confiabilidad
50,ESP,2.70,3.00,0.66,2.0,3.0,91.6%,99.9%,100.0%,99.3%,0.7%,Prioridad robusta,Alta confiabilidad
51,CHL,4.52,4.00,0.77,4.0,5.0,5.8%,91.8%,100.0%,51.4%,48.6%,Prioridad condicional,Confiabilidad media
52,CRI,4.92,5.00,1.37,4.0,7.0,7.9%,74.0%,99.8%,45.6%,53.7%,Prioridad condicional,Confiabilidad media
53,ECU,6.83,6.00,1.64,5.0,9.0,0.3%,16.3%,97.3%,1.8%,91.1%,Prioridad condicional,Confiabilidad media
54,USA,7.71,7.00,2.57,5.0,12.0,0.3%,15.4%,86.0%,2.4%,77.3%,Prioridad condicional,Confiabilidad media
61,BHS,13.10,13.00,3.67,8.0,18.0,0.0%,0.9%,29.5%,0.0%,20.6%,Prioridad no robusta,No prioritario / baja evidencia
55,MEX,8.60,8.00,1.67,7.0,11.0,0.0%,0.8%,86.9%,0.0%,74.5%,Prioridad condicional,Confiabilidad media
56,CAN,10.40,10.00,3.43,7.0,16.0,0.0%,0.6%,59.6%,0.1%,48.8%,Prioridad no robusta,No prioritario / baja evidencia


### IB — Top probabilístico RADAR

,country_iso3,mean_rank,median_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
72,PAN,1.02,1.00,0.14,1.0,1.0,100.0%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
73,CRI,2.24,2.00,0.45,2.0,3.0,99.9%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
74,ESP,3.01,3.00,0.72,2.0,4.0,77.7%,99.9%,100.0%,98.1%,1.9%,Prioridad robusta,Alta confiabilidad
75,DOM,4.14,4.00,1.01,3.0,5.0,22.1%,92.0%,100.0%,74.0%,25.9%,Prioridad robusta,Alta confiabilidad
76,CHL,4.93,5.00,0.77,4.0,6.0,0.3%,87.0%,100.0%,25.3%,74.7%,Prioridad condicional,Confiabilidad media
77,USA,7.97,7.00,2.40,6.0,11.0,0.0%,8.0%,81.8%,1.3%,72.3%,Prioridad condicional,Confiabilidad media
79,ECU,8.60,8.00,2.56,6.0,12.0,0.0%,5.5%,72.6%,0.8%,64.5%,Prioridad no robusta,No prioritario / baja evidencia
78,HND,8.52,8.00,2.09,6.0,11.0,0.0%,3.9%,81.7%,0.2%,67.2%,Prioridad no robusta,No prioritario / baja evidencia
83,BHS,11.31,12.00,3.17,7.0,16.0,0.0%,2.8%,41.1%,0.3%,32.8%,Prioridad no robusta,No prioritario / baja evidencia
82,GTM,10.34,11.00,1.97,8.0,13.0,0.0%,0.5%,49.9%,0.0%,34.7%,Prioridad no robusta,No prioritario / baja evidencia


### PF — Top probabilístico RADAR

,country_iso3,mean_rank,median_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
96,PAN,1.31,1.00,0.47,1.0,2.0,100.0%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
98,CRI,2.79,3.00,0.43,2.0,3.0,99.2%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
97,ESP,1.97,2.00,0.83,1.0,3.0,95.1%,100.0%,100.0%,99.8%,0.2%,Prioridad robusta,Alta confiabilidad
99,DOM,4.16,4.00,0.60,4.0,5.0,5.7%,97.6%,100.0%,79.9%,20.1%,Prioridad robusta,Alta confiabilidad
100,CHL,5.05,5.00,0.97,4.0,6.0,0.0%,87.2%,99.7%,19.8%,78.8%,Prioridad condicional,Confiabilidad media
103,SLV,9.14,9.00,2.88,6.0,13.0,0.0%,8.8%,65.6%,0.2%,56.3%,Prioridad no robusta,No prioritario / baja evidencia
102,ECU,8.57,8.00,2.27,6.0,11.0,0.0%,2.9%,80.4%,0.1%,66.2%,Prioridad no robusta,No prioritario / baja evidencia
101,URY,7.18,7.00,1.62,6.0,9.0,0.0%,2.0%,93.7%,0.1%,91.8%,Prioridad condicional,Confiabilidad media
104,USA,9.55,9.00,3.32,6.0,15.0,0.0%,1.4%,66.0%,0.1%,57.7%,Prioridad no robusta,No prioritario / baja evidencia
109,HND,13.72,15.00,3.20,9.0,17.0,0.0%,0.1%,19.2%,0.0%,14.5%,Prioridad no robusta,No prioritario / baja evidencia


In [15]:

# ---------------------------------------------------------------------------
# Lectura ejecutiva automática: líderes robustos por línea
# ---------------------------------------------------------------------------
leader_rows = []
for line in sorted(radar_robustness_exec["business_line"].unique()):
    tmp = radar_robustness_exec.query("business_line == @line").copy()
    tmp = tmp.sort_values(["prob_top_5", "prob_top_10", "Alta", "mean_rank"], ascending=[False, False, False, True])
    leader = tmp.iloc[0]
    leader_rows.append({
        "business_line": line,
        "country_iso3": leader["country_iso3"],
        "mean_rank": leader.get("mean_rank", np.nan),
        "std_rank": leader.get("std_rank", np.nan),
        "prob_top_5": leader.get("prob_top_5", np.nan),
        "prob_top_10": leader.get("prob_top_10", np.nan),
        "prob_alta": leader.get("Alta", np.nan),
        "robustness_class": leader["robustness_class"],
        "confidence_label": leader["confidence_label"],
    })

leaders_df = pd.DataFrame(leader_rows)
display(style_table(
    leaders_df,
    gradient_cols=["mean_rank", "std_rank", "prob_top_5", "prob_top_10", "prob_alta"],
    gradient_cmap="RdYlGn",
    format_dict={"mean_rank": "{:.2f}", "std_rank": "{:.2f}", "prob_top_5": "{:.1%}", "prob_top_10": "{:.1%}", "prob_alta": "{:.1%}"},
).set_caption("Líder probabilístico por línea de negocio"))

for _, row in leaders_df.iterrows():
    display(Markdown(f"""
### Lectura ejecutiva — {row['business_line']}

El país con mayor evidencia probabilística para **{row['business_line']}** es **{row['country_iso3']}**.  
Tiene una probabilidad de **Top 5 de {row['prob_top_5']:.1%}**, probabilidad de **Top 10 de {row['prob_top_10']:.1%}** y volatilidad de ranking de **{row['std_rank']:.2f}**.  
Clasificación: **{row['robustness_class']}** ({row['confidence_label']}).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.
"""))


,business_line,country_iso3,mean_rank,std_rank,prob_top_5,prob_top_10,prob_alta,robustness_class,confidence_label
0,AD,CRI,1.67,0.64,100.0%,100.0%,100.0%,Prioridad robusta,Alta confiabilidad
1,BD,ESP,2.26,1.03,100.0%,100.0%,100.0%,Prioridad robusta,Alta confiabilidad
2,CIB,PAN,1.02,0.18,100.0%,100.0%,99.9%,Prioridad robusta,Alta confiabilidad
3,IB,PAN,1.02,0.14,100.0%,100.0%,100.0%,Prioridad robusta,Alta confiabilidad
4,PF,PAN,1.31,0.47,100.0%,100.0%,100.0%,Prioridad robusta,Alta confiabilidad



### Lectura ejecutiva — AD

El país con mayor evidencia probabilística para **AD** es **CRI**.  
Tiene una probabilidad de **Top 5 de 100.0%**, probabilidad de **Top 10 de 100.0%** y volatilidad de ranking de **0.64**.  
Clasificación: **Prioridad robusta** (Alta confiabilidad).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.



### Lectura ejecutiva — BD

El país con mayor evidencia probabilística para **BD** es **ESP**.  
Tiene una probabilidad de **Top 5 de 100.0%**, probabilidad de **Top 10 de 100.0%** y volatilidad de ranking de **1.03**.  
Clasificación: **Prioridad robusta** (Alta confiabilidad).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.



### Lectura ejecutiva — CIB

El país con mayor evidencia probabilística para **CIB** es **PAN**.  
Tiene una probabilidad de **Top 5 de 100.0%**, probabilidad de **Top 10 de 100.0%** y volatilidad de ranking de **0.18**.  
Clasificación: **Prioridad robusta** (Alta confiabilidad).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.



### Lectura ejecutiva — IB

El país con mayor evidencia probabilística para **IB** es **PAN**.  
Tiene una probabilidad de **Top 5 de 100.0%**, probabilidad de **Top 10 de 100.0%** y volatilidad de ranking de **0.14**.  
Clasificación: **Prioridad robusta** (Alta confiabilidad).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.



### Lectura ejecutiva — PF

El país con mayor evidencia probabilística para **PF** es **PAN**.  
Tiene una probabilidad de **Top 5 de 100.0%**, probabilidad de **Top 10 de 100.0%** y volatilidad de ranking de **0.47**.  
Clasificación: **Prioridad robusta** (Alta confiabilidad).

**Implicación:** si esta clasificación es robusta, puede avanzar a análisis comercial/regulatorio. Si es condicional, debe acompañarse de validación cualitativa y revisión de drivers.



## 11. Probabilidad Top-N: la métrica más defendible ante Comité

El ranking puntual es frágil cuando hay empates prácticos. La probabilidad Top-N responde mejor a la pregunta ejecutiva:

> **¿Qué tan probable es que este país siga siendo prioritario si aceptamos incertidumbre razonable en los pesos?**

Interpretación sugerida:

- `prob_top_5 >= 80%`: prioridad robusta para discusión ejecutiva.
- `prob_top_10 >= 70%`: prioridad condicional o shortlist ampliada.
- `prob_top_5 < 50%` con buen ranking base: señal de fragilidad.


In [16]:

for business_line in sorted(radar_topn["business_line"].unique()):
    tmp = radar_topn.query("business_line == @business_line").sort_values("prob_top_5", ascending=False).head(15)
    display(Markdown(f"### {business_line} — Probabilidad Top-N RADAR"))
    display(style_table(
        tmp,
        gradient_cols=["prob_top_3", "prob_top_5", "prob_top_10", "prob_top_15"],
        gradient_cmap="RdYlGn",
        format_dict={"prob_top_3": "{:.1%}", "prob_top_5": "{:.1%}", "prob_top_10": "{:.1%}", "prob_top_15": "{:.1%}"},
    ))


### AD — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
0,AD,CRI,98.2%,100.0%,100.0%,100.0%
1,AD,ESP,93.6%,100.0%,100.0%,100.0%
2,AD,PAN,92.4%,99.6%,100.0%,100.0%
3,AD,USA,15.7%,93.4%,100.0%,100.0%
4,AD,CHL,0.0%,82.4%,100.0%,100.0%
5,AD,DOM,0.1%,22.7%,100.0%,100.0%
6,AD,URY,0.0%,1.0%,100.0%,100.0%
7,AD,CAN,0.0%,0.8%,98.6%,100.0%
8,AD,BHS,0.0%,0.1%,80.8%,99.8%
16,AD,JAM,0.0%,0.0%,0.0%,0.2%


### BD — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
24,BD,CRI,97.5%,100.0%,100.0%,100.0%
25,BD,ESP,87.7%,100.0%,100.0%,100.0%
26,BD,PAN,91.0%,99.1%,100.0%,100.0%
27,BD,DOM,19.1%,96.2%,100.0%,100.0%
28,BD,CHL,2.3%,82.2%,100.0%,100.0%
29,BD,USA,2.3%,19.5%,97.2%,100.0%
30,BD,MEX,0.1%,2.4%,98.3%,100.0%
31,BD,ECU,0.0%,0.6%,74.0%,99.8%
40,BD,JAM,0.0%,0.0%,0.0%,0.0%
46,BD,TTO,0.0%,0.0%,0.0%,0.0%


### CIB — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
48,CIB,DOM,94.2%,100.0%,100.0%,100.0%
49,CIB,PAN,99.9%,100.0%,100.0%,100.0%
50,CIB,ESP,91.6%,99.9%,100.0%,100.0%
51,CIB,CHL,5.8%,91.8%,100.0%,100.0%
52,CIB,CRI,7.9%,74.0%,99.8%,100.0%
53,CIB,ECU,0.3%,16.3%,97.3%,100.0%
54,CIB,USA,0.3%,15.4%,86.0%,99.0%
55,CIB,BHS,0.0%,0.9%,29.5%,68.7%
56,CIB,MEX,0.0%,0.8%,86.9%,100.0%
57,CIB,CAN,0.0%,0.6%,59.6%,89.4%


### IB — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
72,IB,CRI,99.9%,100.0%,100.0%,100.0%
73,IB,PAN,100.0%,100.0%,100.0%,100.0%
74,IB,ESP,77.7%,99.9%,100.0%,100.0%
75,IB,DOM,22.1%,92.0%,100.0%,100.0%
76,IB,CHL,0.3%,87.0%,100.0%,100.0%
77,IB,USA,0.0%,8.0%,81.8%,99.4%
78,IB,ECU,0.0%,5.5%,72.6%,99.9%
79,IB,HND,0.0%,3.9%,81.7%,99.8%
80,IB,BHS,0.0%,2.8%,41.1%,88.2%
81,IB,GTM,0.0%,0.5%,49.9%,99.8%


### PF — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
96,PF,CRI,99.2%,100.0%,100.0%,100.0%
98,PF,PAN,100.0%,100.0%,100.0%,100.0%
97,PF,ESP,95.1%,100.0%,100.0%,100.0%
99,PF,DOM,5.7%,97.6%,100.0%,100.0%
100,PF,CHL,0.0%,87.2%,99.7%,100.0%
101,PF,SLV,0.0%,8.8%,65.6%,99.2%
102,PF,ECU,0.0%,2.9%,80.4%,99.3%
103,PF,URY,0.0%,2.0%,93.7%,100.0%
104,PF,USA,0.0%,1.4%,66.0%,90.8%
105,PF,HND,0.0%,0.1%,19.2%,65.7%



## 12. Probabilidad por banda: estabilidad de atractividad, no solo posición

Un país puede no ser Top 5 estable, pero sí mantenerse en banda alta o media-alta. Eso es relevante para oportunidades exploratorias, alianzas o monitoreo estratégico.

**Lectura ejecutiva:**

- Alta probabilidad en banda **Alta**: atractivo estructuralmente consistente.
- Alta probabilidad en **Alta + Media-alta**: oportunidad defendible, aunque el ranking exacto fluctúe.
- Alta probabilidad en banda **Media/Baja**: no priorizar sin argumento estratégico externo.


In [17]:

for business_line in sorted(radar_tiers["business_line"].unique()):
    tmp = radar_tiers.query("business_line == @business_line").sort_values(["Alta", "Media-alta"], ascending=False).head(15)
    display(Markdown(f"### {business_line} — Probabilidad por banda RADAR"))
    display(style_table(
        tmp,
        gradient_cols=["Alta", "Media-alta", "Media", "Baja"],
        gradient_cmap="RdYlGn",
        format_dict={"Alta": "{:.1%}", "Media-alta": "{:.1%}", "Media": "{:.1%}", "Baja": "{:.1%}"},
    ))


### AD — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
7,AD,CRI,100.0%,0.0%,0.0%,0.0%
10,AD,ESP,100.0%,0.0%,0.0%,0.0%
16,AD,PAN,98.6%,1.4%,0.0%,0.0%
23,AD,USA,80.3%,19.7%,0.0%,0.0%
8,AD,DOM,10.6%,89.4%,0.0%,0.0%
6,AD,CHL,10.4%,89.6%,0.0%,0.0%
5,AD,CAN,0.1%,97.4%,2.5%,0.0%
22,AD,URY,0.0%,100.0%,0.0%,0.0%
1,AD,BHS,0.0%,70.2%,28.5%,1.3%
17,AD,PER,0.0%,22.7%,77.3%,0.0%


### BD — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
34,BD,ESP,100.0%,0.0%,0.0%,0.0%
31,BD,CRI,99.6%,0.4%,0.0%,0.0%
40,BD,PAN,96.6%,3.4%,0.0%,0.0%
32,BD,DOM,86.0%,14.0%,0.0%,0.0%
30,BD,CHL,12.2%,87.8%,0.0%,0.0%
47,BD,USA,5.5%,88.6%,5.9%,0.0%
38,BD,MEX,0.1%,93.4%,6.5%,0.0%
41,BD,PER,0.0%,60.3%,39.6%,0.1%
46,BD,URY,0.0%,59.6%,40.0%,0.4%
33,BD,ECU,0.0%,59.3%,40.1%,0.6%


### CIB — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
64,CIB,PAN,99.9%,0.1%,0.0%,0.0%
56,CIB,DOM,99.5%,0.5%,0.0%,0.0%
58,CIB,ESP,99.3%,0.7%,0.0%,0.0%
54,CIB,CHL,51.4%,48.6%,0.0%,0.0%
55,CIB,CRI,45.6%,53.7%,0.7%,0.0%
71,CIB,USA,2.4%,77.3%,17.9%,2.4%
57,CIB,ECU,1.8%,91.1%,7.1%,0.0%
53,CIB,CAN,0.1%,48.8%,34.6%,16.5%
62,CIB,MEX,0.0%,74.5%,25.2%,0.3%
65,CIB,PER,0.0%,31.0%,62.9%,6.1%


### IB — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
79,IB,CRI,100.0%,0.0%,0.0%,0.0%
88,IB,PAN,100.0%,0.0%,0.0%,0.0%
82,IB,ESP,98.1%,1.9%,0.0%,0.0%
80,IB,DOM,74.0%,25.9%,0.1%,0.0%
78,IB,CHL,25.3%,74.7%,0.0%,0.0%
95,IB,USA,1.3%,72.3%,25.1%,1.3%
81,IB,ECU,0.8%,64.5%,34.1%,0.6%
73,IB,BHS,0.3%,32.8%,49.1%,17.8%
84,IB,HND,0.2%,67.2%,32.2%,0.4%
94,IB,URY,0.0%,63.4%,36.2%,0.4%


### PF — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
103,PF,CRI,100.0%,0.0%,0.0%,0.0%
112,PF,PAN,100.0%,0.0%,0.0%,0.0%
106,PF,ESP,99.8%,0.2%,0.0%,0.0%
104,PF,DOM,79.9%,20.1%,0.0%,0.0%
102,PF,CHL,19.8%,78.8%,1.4%,0.0%
115,PF,SLV,0.2%,56.3%,40.3%,3.2%
118,PF,URY,0.1%,91.8%,8.0%,0.1%
105,PF,ECU,0.1%,66.2%,32.1%,1.6%
119,PF,USA,0.1%,57.7%,28.1%,14.1%
96,PF,ARG,0.0%,58.6%,32.6%,8.8%



## 13. Volatilidad del ranking: dónde el modelo es confiable y dónde es frágil

La volatilidad (`std_rank`) mide cuántas posiciones puede moverse un país bajo perturbaciones de pesos. En la práctica:

- baja volatilidad en Top 5 → prioridad confiable;
- alta volatilidad en Top 5 → decisión sensible;
- alta volatilidad en países medios → normal, porque hay muchos empates prácticos;
- volatilidad alta + baja probabilidad Top-N → no usar como prioridad.


In [18]:

volatility_summary = (
    radar_robustness_exec
    .groupby("business_line")
    .agg(
        mean_std_rank=("std_rank", "mean"),
        max_std_rank=("std_rank", "max"),
        median_prob_top_5=("prob_top_5", "median"),
        n_prioridad_robusta=("robustness_class", lambda s: (s == "Prioridad robusta").sum()),
        n_alta_sensibilidad=("robustness_class", lambda s: (s == "Alta sensibilidad").sum()),
    )
    .reset_index()
)

display(style_table(
    volatility_summary,
    gradient_cols=["mean_std_rank", "max_std_rank", "median_prob_top_5", "n_prioridad_robusta", "n_alta_sensibilidad"],
    gradient_cmap="YlOrRd",
    format_dict={"mean_std_rank": "{:.2f}", "max_std_rank": "{:.2f}", "median_prob_top_5": "{:.1%}"},
).set_caption("Resumen de volatilidad y confiabilidad por línea"))

most_volatile = (
    radar_robustness_exec
    .sort_values("std_rank", ascending=False)
    .head(20)
)

display(style_table(
    most_volatile[["business_line", "country_iso3", "mean_rank", "std_rank", "prob_top_5", "prob_top_10", "Alta", "robustness_class"]],
    gradient_cols=["mean_rank", "std_rank", "prob_top_5", "prob_top_10", "Alta"],
    gradient_cmap="YlOrRd",
    format_dict={"mean_rank": "{:.2f}", "std_rank": "{:.2f}", "prob_top_5": "{:.1%}", "prob_top_10": "{:.1%}", "Alta": "{:.1%}"},
).set_caption("Países/líneas con mayor volatilidad de ranking"))


,business_line,mean_std_rank,max_std_rank,median_prob_top_5,n_prioridad_robusta,n_alta_sensibilidad
0,AD,1.09,2.70,0.0%,4,0
1,BD,1.42,2.77,0.0%,4,0
2,CIB,1.58,3.67,0.1%,3,0
3,IB,1.56,3.23,0.0%,4,0
4,PF,1.81,3.71,0.0%,4,0


,business_line,country_iso3,mean_rank,std_rank,prob_top_5,prob_top_10,Alta,robustness_class
108,PF,CAN,13.65,3.71,0.0%,25.3%,0.0%,Prioridad no robusta
61,CIB,BHS,13.10,3.67,0.9%,29.5%,0.0%,Prioridad no robusta
56,CIB,CAN,10.40,3.43,0.6%,59.6%,0.1%,Prioridad no robusta
104,PF,USA,9.55,3.32,1.4%,66.0%,0.1%,Prioridad no robusta
84,IB,CAN,13.91,3.23,0.0%,17.2%,0.0%,Prioridad no robusta
109,PF,HND,13.72,3.20,0.1%,19.2%,0.0%,Prioridad no robusta
83,IB,BHS,11.31,3.17,2.8%,41.1%,0.3%,Prioridad no robusta
111,PF,GTM,15.09,3.08,0.0%,10.3%,0.0%,Prioridad no robusta
112,PF,NIC,16.55,3.05,0.0%,6.8%,0.0%,Prioridad no robusta
103,PF,SLV,9.14,2.88,8.8%,65.6%,0.2%,Prioridad no robusta



## 14. Estabilidad de correlaciones entre líneas

La correlación entre líneas permite saber si dos tesis se comportan de forma parecida. Monte Carlo permite saber si esa cercanía es estable o depende de pesos específicos.

**Lectura ejecutiva:**

- Correlación alta y estable: las líneas comparten un factor país estructural.
- Correlación alta pero inestable: la similitud depende demasiado de pesos.
- Correlación baja y estable: tesis claramente diferenciadas.


In [19]:

line_corr_radar = mc_radar["line_correlation_robustness"]

display(style_table(
    line_corr_radar,
    gradient_cols=["mean_spearman", "p10_spearman", "p90_spearman", "std_spearman"],
    gradient_cmap="YlOrRd",
    format_dict={
        "mean_spearman": "{:.3f}",
        "median_spearman": "{:.3f}",
        "std_spearman": "{:.3f}",
        "p10_spearman": "{:.3f}",
        "p90_spearman": "{:.3f}",
        "min_spearman": "{:.3f}",
        "max_spearman": "{:.3f}",
    },
).set_caption("Robustez de correlaciones entre líneas — Monte Carlo RADAR"))

stable_high_corr = line_corr_radar.query("mean_spearman >= 0.80 and std_spearman <= 0.05") if "std_spearman" in line_corr_radar.columns else pd.DataFrame()
if not stable_high_corr.empty:
    insight_box(
        "Atractivo común estable entre líneas",
        "Algunas líneas mantienen alta correlación en Monte Carlo. Esto no implica redundancia necesariamente: puede indicar que los mismos países concentran atributos estructurales valiosos para varias tesis de negocio."
    )


,line_a,line_b,mean_spearman,median_spearman,std_spearman,p10_spearman,p90_spearman,min_spearman,max_spearman
0,PF,BD,0.894,0.915,0.066,0.809,0.950,0.544,0.980
1,IB,CIB,0.889,0.891,0.038,0.839,0.937,0.717,0.985
2,AD,BD,0.859,0.863,0.036,0.812,0.902,0.710,0.941
3,BD,CIB,0.839,0.843,0.049,0.776,0.897,0.523,0.957
4,IB,PF,0.835,0.838,0.051,0.769,0.897,0.658,0.961
5,PF,CIB,0.822,0.824,0.051,0.761,0.885,0.613,0.974
6,PF,AD,0.817,0.838,0.072,0.720,0.885,0.472,0.926
7,AD,CIB,0.799,0.803,0.060,0.719,0.870,0.459,0.937
8,IB,BD,0.797,0.800,0.054,0.723,0.862,0.600,0.928
9,IB,AD,0.785,0.791,0.066,0.697,0.867,0.523,0.951



## 15. Distribución de alpha, beta y gamma: robustez de la mezcla RADAR

La mezcla `alpha`, `beta`, `gamma` también es un supuesto. Si pequeñas perturbaciones en esa mezcla cambian radicalmente el ranking, la decisión sería frágil.

La distribución simulada permite revisar si la priorización depende excesivamente de una composición puntual entre estructura, proximidad y tendencia.


In [20]:

composite_distribution = mc_radar.get("composite_weight_distribution")
if composite_distribution is not None and not composite_distribution.empty:
    display(style_table(composite_distribution).set_caption("Distribución simulada de pesos compuestos alpha, beta, gamma"))
else:
    insight_box("Pesos compuestos no disponibles", "La salida de Monte Carlo no contiene distribución de alpha, beta y gamma. Revise si perturb_composite_weights está activo.")


,alpha,beta,gamma
mean,0.599479,0.300524,0.099998
std,0.039311,0.036891,0.024279
min,0.472461,0.202170,0.044878
max,0.726835,0.426420,0.176576



## 16. Tabla ejecutiva final por línea: ranking base + robustez probabilística

Esta tabla conecta el ranking determinístico con la evidencia de Monte Carlo. Es la tabla más útil para toma de decisiones porque permite diferenciar:

- países altos y robustos;
- países altos pero sensibles;
- países medios con probabilidad alta de banda atractiva;
- países que no deberían avanzar sin validación adicional.


In [21]:

def build_base_radar_by_line(radar_by_line) -> Dict[str, pd.DataFrame]:
    """Convierte results['radar_by_line'] en DataFrames por línea."""
    out: Dict[str, pd.DataFrame] = {}

    if isinstance(radar_by_line, pd.DataFrame):
        df = radar_by_line.copy()
        if "country_iso3" not in df.columns:
            df = df.reset_index().rename(columns={"index": "country_iso3"})
        for col in [c for c in df.columns if c != "country_iso3"]:
            tmp = df[["country_iso3", col]].rename(columns={col: "radar_score"}).copy()
            tmp["radar_score"] = pd.to_numeric(tmp["radar_score"], errors="coerce")
            tmp = tmp.dropna(subset=["radar_score"])
            tmp["base_rank"] = tmp["radar_score"].rank(ascending=False, method="min").astype(int)
            out[col] = tmp.sort_values("base_rank")
        return out

    if isinstance(radar_by_line, dict):
        for line, obj in radar_by_line.items():
            if isinstance(obj, pd.DataFrame):
                tmp = obj.copy()
                if "country_iso3" not in tmp.columns:
                    tmp = tmp.reset_index().rename(columns={"index": "country_iso3"})
                score_col = next((c for c in ["radar_score", "score", "final_score", line, "value"] if c in tmp.columns), None)
                if score_col is None:
                    continue
                tmp = tmp[["country_iso3", score_col]].rename(columns={score_col: "radar_score"})
            elif isinstance(obj, pd.Series):
                tmp = obj.rename("radar_score").reset_index()
                if tmp.columns[0] != "country_iso3":
                    tmp = tmp.rename(columns={tmp.columns[0]: "country_iso3"})
            else:
                continue
            tmp["radar_score"] = pd.to_numeric(tmp["radar_score"], errors="coerce")
            tmp = tmp.dropna(subset=["radar_score"])
            tmp["base_rank"] = tmp["radar_score"].rank(ascending=False, method="min").astype(int)
            out[line] = tmp.sort_values("base_rank")
        return out

    raise TypeError(f"Tipo no soportado para radar_by_line: {type(radar_by_line)}")

base_radar_by_line = build_base_radar_by_line(results["radar_by_line"])

mc_exec_by_line = {}
for line, base_df in base_radar_by_line.items():
    robustness = radar_robustness_exec.query("business_line == @line").copy()
    out = base_df.merge(robustness, on="country_iso3", how="left")
    mc_exec_by_line[line] = out.sort_values("base_rank")

for line, exec_table in mc_exec_by_line.items():
    display(Markdown(f"### {line} — Tabla ejecutiva final"))
    cols = [
        "country_iso3", "radar_score", "base_rank", "mean_rank", "std_rank", "p10_rank", "p90_rank",
        "prob_top_3", "prob_top_5", "prob_top_10", "Alta", "Media-alta", "robustness_class", "confidence_label",
    ]
    cols = [c for c in cols if c in exec_table.columns]
    display(style_table(
        exec_table[cols].head(15),
        gradient_cols=["radar_score", "mean_rank", "std_rank", "prob_top_5", "prob_top_10", "Alta"],
        gradient_cmap="RdYlGn",
        format_dict={
            "radar_score": "{:.3f}",
            "base_rank": "{:.0f}",
            "mean_rank": "{:.2f}",
            "std_rank": "{:.2f}",
            "p10_rank": "{:.1f}",
            "p90_rank": "{:.1f}",
            "prob_top_3": "{:.1%}",
            "prob_top_5": "{:.1%}",
            "prob_top_10": "{:.1%}",
            "Alta": "{:.1%}",
            "Media-alta": "{:.1%}",
        },
    ))


### IB — Tabla ejecutiva final

,country_iso3,radar_score,base_rank,mean_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,PAN,0.698,1,1.02,0.14,1.0,1.0,100.0%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
1,CRI,0.649,2,2.24,0.45,2.0,3.0,99.9%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
2,ESP,0.634,3,3.01,0.72,2.0,4.0,77.7%,99.9%,100.0%,98.1%,1.9%,Prioridad robusta,Alta confiabilidad
3,DOM,0.604,4,4.14,1.01,3.0,5.0,22.1%,92.0%,100.0%,74.0%,25.9%,Prioridad robusta,Alta confiabilidad
4,CHL,0.587,5,4.93,0.77,4.0,6.0,0.3%,87.0%,100.0%,25.3%,74.7%,Prioridad condicional,Confiabilidad media
5,USA,0.549,6,7.97,2.40,6.0,11.0,0.0%,8.0%,81.8%,1.3%,72.3%,Prioridad condicional,Confiabilidad media
6,HND,0.543,7,8.52,2.09,6.0,11.0,0.0%,3.9%,81.7%,0.2%,67.2%,Prioridad no robusta,No prioritario / baja evidencia
7,ECU,0.542,8,8.60,2.56,6.0,12.0,0.0%,5.5%,72.6%,0.8%,64.5%,Prioridad no robusta,No prioritario / baja evidencia
8,URY,0.542,9,8.68,2.18,6.0,12.0,0.0%,0.4%,77.6%,0.0%,63.4%,Prioridad no robusta,No prioritario / baja evidencia
9,SLV,0.534,10,9.68,1.45,8.0,12.0,0.0%,0.0%,69.3%,0.0%,46.6%,Prioridad no robusta,No prioritario / baja evidencia


### PF — Tabla ejecutiva final

,country_iso3,radar_score,base_rank,mean_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,PAN,0.667,1,1.31,0.47,1.0,2.0,100.0%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
1,ESP,0.653,2,1.97,0.83,1.0,3.0,95.1%,100.0%,100.0%,99.8%,0.2%,Prioridad robusta,Alta confiabilidad
2,CRI,0.635,3,2.79,0.43,2.0,3.0,99.2%,100.0%,100.0%,100.0%,0.0%,Prioridad robusta,Alta confiabilidad
3,DOM,0.596,4,4.16,0.60,4.0,5.0,5.7%,97.6%,100.0%,79.9%,20.1%,Prioridad robusta,Alta confiabilidad
4,CHL,0.570,5,5.05,0.97,4.0,6.0,0.0%,87.2%,99.7%,19.8%,78.8%,Prioridad condicional,Confiabilidad media
5,URY,0.531,6,7.18,1.62,6.0,9.0,0.0%,2.0%,93.7%,0.1%,91.8%,Prioridad condicional,Confiabilidad media
6,ECU,0.517,7,8.57,2.27,6.0,11.0,0.0%,2.9%,80.4%,0.1%,66.2%,Prioridad no robusta,No prioritario / baja evidencia
7,USA,0.515,8,9.55,3.32,6.0,15.0,0.0%,1.4%,66.0%,0.1%,57.7%,Prioridad no robusta,No prioritario / baja evidencia
8,ARG,0.512,9,9.86,2.56,8.0,14.0,0.0%,0.0%,73.3%,0.0%,58.6%,Prioridad no robusta,No prioritario / baja evidencia
9,SLV,0.512,10,9.14,2.88,6.0,13.0,0.0%,8.8%,65.6%,0.2%,56.3%,Prioridad no robusta,No prioritario / baja evidencia


### AD — Tabla ejecutiva final

...

,country_iso3,radar_score,base_rank,mean_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,PAN,0.670,1,1.02,0.18,1.0,1.0,99.9%,100.0%,100.0%,99.9%,0.1%,Prioridad robusta,Alta confiabilidad
1,DOM,0.625,2,2.45,0.63,2.0,3.0,94.2%,100.0%,100.0%,99.5%,0.5%,Prioridad robusta,Alta confiabilidad
2,ESP,0.618,3,2.70,0.66,2.0,3.0,91.6%,99.9%,100.0%,99.3%,0.7%,Prioridad robusta,Alta confiabilidad
3,CHL,0.583,4,4.52,0.77,4.0,5.0,5.8%,91.8%,100.0%,51.4%,48.6%,Prioridad condicional,Confiabilidad media
4,CRI,0.576,5,4.92,1.37,4.0,7.0,7.9%,74.0%,99.8%,45.6%,53.7%,Prioridad condicional,Confiabilidad media
5,ECU,0.546,6,6.83,1.64,5.0,9.0,0.3%,16.3%,97.3%,1.8%,91.1%,Prioridad condicional,Confiabilidad media
6,USA,0.542,7,7.71,2.57,5.0,12.0,0.3%,15.4%,86.0%,2.4%,77.3%,Prioridad condicional,Confiabilidad media
7,MEX,0.526,8,8.60,1.67,7.0,11.0,0.0%,0.8%,86.9%,0.0%,74.5%,Prioridad condicional,Confiabilidad media
8,CAN,0.522,9,10.40,3.43,7.0,16.0,0.0%,0.6%,59.6%,0.1%,48.8%,Prioridad no robusta,No prioritario / baja evidencia
9,PER,0.512,10,10.70,2.10,8.0,14.0,0.0%,0.0%,52.0%,0.0%,31.0%,Prioridad no robusta,No prioritario / baja evidencia


### GLOBAL — Tabla ejecutiva final

,country_iso3,radar_score,base_rank,mean_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,PAN,0.686,1,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
1,CRI,0.641,2,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
2,DOM,0.629,3,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
3,ESP,0.611,4,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
4,CHL,0.576,5,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
5,USA,0.554,6,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
6,URY,0.542,7,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
7,MEX,0.535,8,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
8,GTM,0.532,9,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
9,PER,0.520,10,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan


### rank_global — Tabla ejecutiva final

,country_iso3,radar_score,base_rank,mean_rank,std_rank,p10_rank,p90_rank,prob_top_3,prob_top_5,prob_top_10,Alta,Media-alta,robustness_class,confidence_label
0,SUR,24.000,1,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
1,BLZ,23.000,2,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
2,TTO,22.000,3,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
3,JAM,21.000,4,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
4,BRA,20.000,5,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
5,BOL,19.000,6,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
6,ARG,18.000,7,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
7,NIC,17.000,8,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
8,BHS,16.000,9,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan
9,PRY,15.000,10,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan,nan



## 17. ¿El modelo es robusto? Respuesta ejecutiva basada en evidencia

La robustez del modelo no se evalúa con una sola métrica. Se evalúa con una combinación de estabilidad estructural, estabilidad decisional y consistencia probabilística.

Criterios propuestos:

- **Modelo confiable para decisión:** existen prioridades robustas por línea, baja volatilidad en líderes, alta probabilidad Top-N y bandas estables.
- **Modelo útil pero condicionado:** hay prioridades claras, pero con países/líneas sensibles que requieren revisión cualitativa.
- **Modelo frágil:** el Top-N cambia sustancialmente, no hay países con probabilidad alta de permanecer en banda superior o la volatilidad es alta en líderes.


In [22]:

# ---------------------------------------------------------------------------
# Diagnóstico global de confiabilidad del modelo
# ---------------------------------------------------------------------------
total_pairs = len(radar_robustness_exec)
robust_count = (radar_robustness_exec["robustness_class"] == "Prioridad robusta").sum()
conditional_count = (radar_robustness_exec["robustness_class"] == "Prioridad condicional").sum()
sensitive_count = (radar_robustness_exec["robustness_class"] == "Alta sensibilidad").sum()

share_robust_or_conditional = (robust_count + conditional_count) / total_pairs if total_pairs else 0
share_sensitive = sensitive_count / total_pairs if total_pairs else 0

if share_robust_or_conditional >= 0.30 and share_sensitive <= 0.25:
    model_reliability = "Confiabilidad suficiente para priorización ejecutiva"
elif share_robust_or_conditional >= 0.15:
    model_reliability = "Confiabilidad útil pero condicionada"
else:
    model_reliability = "Confiabilidad limitada; requiere recalibración o validación adicional"

reliability_df = pd.DataFrame({
    "métrica": [
        "pares_país_línea",
        "prioridades_robustas",
        "prioridades_condicionales",
        "alta_sensibilidad",
        "share_robustas_o_condicionales",
        "share_alta_sensibilidad",
        "diagnóstico_global",
    ],
    "valor": [
        total_pairs,
        robust_count,
        conditional_count,
        sensitive_count,
        share_robust_or_conditional,
        share_sensitive,
        model_reliability,
    ],
})

display(style_table(reliability_df, format_dict={"valor": "{}"}).set_caption("Diagnóstico global de confiabilidad"))

if "suficiente" in model_reliability:
    success_box("Conclusión de robustez", model_reliability + ". El modelo puede usarse para priorizar, diferenciando prioridades robustas de condicionales.")
elif "condicionada" in model_reliability:
    insight_box("Conclusión de robustez", model_reliability + ". El modelo es útil, pero el Comité debe revisar países sensibles y validar drivers antes de decisiones irreversibles.")
else:
    risk_box("Conclusión de robustez", model_reliability + ". No se recomienda usar el ranking como insumo único de decisión sin recalibración o validación adicional.")


,métrica,valor
0,pares_país_línea,120
1,prioridades_robustas,19
2,prioridades_condicionales,17
3,alta_sensibilidad,0
4,share_robustas_o_condicionales,0.3
5,share_alta_sensibilidad,0.0
6,diagnóstico_global,Confiabilidad suficiente para priorización ejecutiva



## 18. Preguntas esperadas del Comité y respuestas sugeridas

Esta sección anticipa preguntas típicas de Comité Ejecutivo y prepara respuestas basadas en evidencia del notebook.



### Pregunta 1 — ¿El ranking es confiable o cambia demasiado con los pesos?

**Respuesta sugerida:**  
El ranking base no se presenta solo. Se valida con Monte Carlo RADAR, que perturba pesos estructurales y la mezcla `alpha`, `beta`, `gamma`. La confiabilidad se mide con probabilidad Top-N, volatilidad de ranking y probabilidad de banda. Los países clasificados como **Prioridad robusta** mantienen una posición alta bajo escenarios alternativos; los **condicionales** requieren validación adicional.

---

### Pregunta 2 — ¿Por qué usamos Monte Carlo RADAR y no solo TOPSIS?

**Respuesta sugerida:**  
TOPSIS mide atractivo estructural, pero la decisión Cibest incorpora también proximidad con Colombia y momentum macro. Por eso Monte Carlo TOPSIS es diagnóstico estructural, mientras Monte Carlo RADAR es la prueba decisional principal. Si un país es robusto en TOPSIS pero no en RADAR, puede ser atractivo en abstracto pero menos ejecutable para Cibest.

---

### Pregunta 3 — ¿Qué significa que un país tenga 80% de probabilidad Top 5?

**Respuesta sugerida:**  
Significa que en 8 de cada 10 escenarios simulados, bajo perturbaciones razonables de pesos, el país permanece dentro de los cinco mercados más atractivos para esa línea. Es una medida de estabilidad de la recomendación, no una probabilidad de éxito comercial.

---

### Pregunta 4 — ¿Qué hacemos con países que tienen buen ranking base pero baja probabilidad Top 5?

**Respuesta sugerida:**  
No deben descartarse automáticamente, pero deben tratarse como prioridades condicionales o sensibles. La recomendación es revisar drivers, restricciones, calidad de datos y factibilidad regulatoria antes de avanzar a business case.

---

### Pregunta 5 — ¿La alta correlación entre algunas líneas indica que el modelo está duplicando tesis?

**Respuesta sugerida:**  
No necesariamente. Si la correlación entre rankings es alta pero los vectores de pesos son distintos, la explicación más probable es un factor país común: algunos mercados concentran madurez institucional, digital y financiera. Eso los hace atractivos para varias líneas simultáneamente. La redundancia sería preocupante solo si rankings y pesos fueran muy parecidos.

---

### Pregunta 6 — ¿Monte Carlo prueba que el país será exitoso?

**Respuesta sugerida:**  
No. Monte Carlo prueba estabilidad interna del modelo ante incertidumbre de pesos. No reemplaza validaciones comerciales, regulatorias, competitivas o financieras. Su función es separar recomendaciones robustas de recomendaciones frágiles.

---

### Pregunta 7 — ¿Qué decisión habilita este notebook?

**Respuesta sugerida:**  
Permite construir una shortlist por línea diferenciando tres grupos: países robustos para profundizar, países condicionales que requieren validación adicional y países no robustos que no deberían avanzar salvo razón estratégica externa.



## 19. Recomendaciones finales para decisión

1. **Usar Monte Carlo RADAR como prueba estándar antes de publicar rankings ejecutivos.**  
   TOPSIS es útil, pero no captura proximidad ni momentum.

2. **Presentar siempre ranking base con probabilidad Top 5, probabilidad Top 10 y probabilidad de banda alta/media-alta.**  
   Esto reduce el riesgo de sobreinterpretar posiciones puntuales.

3. **Distinguir prioridades robustas de prioridades condicionales.**  
   No todos los países del Top 5 tienen el mismo nivel de confiabilidad.

4. **Usar volatilidad de ranking como alerta temprana.**  
   Alta volatilidad en países líderes debe activar revisión de drivers y restricciones.

5. **Complementar robustez cuantitativa con due diligence cualitativa.**  
   Regulación, competencia, capacidades internas, licencias, partnerships y riesgos país siguen siendo necesarios.

6. **Persistir resultados Monte Carlo para trazabilidad.**  
   Las decisiones deben poder reproducirse y compararse entre corridas.

---

## Síntesis Ejecutiva

El análisis de robustez transforma RADAR de ranking determinístico a sistema probabilístico de decisión. Un país es prioritario no porque aparezca alto una vez, sino porque conserva alta probabilidad de permanecer en Top-N o en banda alta bajo incertidumbre razonable. La recomendación ejecutiva debe separar prioridades robustas, oportunidades condicionales y países sensibles. Monte Carlo RADAR es la evidencia principal de confiabilidad; Monte Carlo TOPSIS es diagnóstico estructural.
